In [0]:
from pyspark.sql import functions as f
from pyspark import StorageLevel

In [0]:
%sql
-- drop table if exists nleshin_catalog.bronze_layer.objects_description_mirror;
-- drop table if exists nleshin_catalog.bronze_layer.objects_description_journal;

-- Step 1: Create tables if they don't exist

CREATE TABLE IF NOT EXISTS nleshin_catalog.bronze_layer.objects_description_mirror (
  file_name STRING NOT NULL PRIMARY KEY,
  file_path STRING,
  length BIGINT,
  content BINARY,
  operation_flag STRING,
  src_inserted_stamp TIMESTAMP,
  sys_inserted_stamp TIMESTAMP,
  job_run_id STRING
);

CREATE TABLE IF NOT EXISTS nleshin_catalog.bronze_layer.objects_description_journal (
  file_name STRING,
  file_path STRING,
  length BIGINT,
  content BINARY,
  operation_flag STRING,
  src_inserted_stamp TIMESTAMP,
  sys_inserted_stamp TIMESTAMP,
  job_run_id STRING
);

In [0]:
# -- Step 2: Read document from nleshin_catalog.raw_layer.objects_description

df_raw = spark.sql(
    """
    select * 
    from nleshin_catalog.raw_layer.objects_description
    where modificationTime >= dateadd(MINUTE, -cast(:lookback_minutes as int), :date_interval_end) and modificationTime < :date_interval_end
    """,
    args={
        "date_interval_end": dbutils.widgets.get("date_interval_end"),
        "lookback_minutes": dbutils.widgets.get("lookback_minutes")
    }
)

df_raw = df_raw.withColumnRenamed("path", "file_path")
df_raw = df_raw.withColumnRenamed("modificationTime", "src_inserted_stamp")
df_raw = df_raw.withColumn("file_name", f.element_at(f.split(f.col("file_path"), "/"), -1))
df_raw = df_raw.withColumn("file_name", f.regexp_extract(f.col("file_name"), r"([A-z].*)__v__\d*", 1))

In [0]:
# -- Step 3: Read data from nleshin_catalog.bronze_layer.objects_description_mirror

df_mirror = spark.sql("select * from nleshin_catalog.bronze_layer.objects_description_mirror")


In [0]:
# -- Step 4: Compare new documents with documents loaded earlier

df_result = df_raw.alias("src").join(other=df_mirror.alias("trg"), 
                                     on=[f.col("src.file_name") == f.col("trg.file_name")],
                                     how="full"
                                    )

df_result = df_result.localCheckpoint(False)

df_insert = (df_result
             .filter(f.col("trg.file_name").isNull())
             .select("src.file_name", "src.file_path", "src.length", "src.content", "src.src_inserted_stamp")
             .withColumn("operation_flag", f.lit("I"))
            )
df_update = (df_result
             .filter(f.col("src.file_name").isNotNull() & f.col("trg.file_name").isNotNull())
             .select("src.file_name", "src.file_path", "src.length", "src.content", "src.src_inserted_stamp")
             .withColumn("operation_flag", f.lit("U"))
            )

df_mirror = (df_result
             .filter(f.col("src.file_name").isNull())
             .select(
               "trg.file_name",
               "trg.file_path",
               "trg.length",
               "trg.content",
               "trg.src_inserted_stamp",
               "trg.operation_flag",
               "trg.sys_inserted_stamp",
               "trg.job_run_id"
              )
            )

df_journal = df_insert.union(df_update)
df_journal = (df_journal
              .withColumn("sys_inserted_stamp", f.current_timestamp())
              .withColumn("job_run_id", f.lit(dbutils.widgets.get("job_run_id")))
            )
df_journal = df_journal.localCheckpoint(False)

df_mirror = df_journal.union(df_mirror)

In [0]:
# -- Step 5: Update data in the mirror table using MERGE

from delta.tables import DeltaTable

target_mirror = DeltaTable.forName(spark, "nleshin_catalog.bronze_layer.objects_description_mirror")

target_mirror.alias("trg").merge(
    source=df_mirror.alias("src"),
    condition="trg.file_name = src.file_name"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

In [0]:
# -- Step 6: Append data to the journal table

df_journal.write.mode("append").saveAsTable("nleshin_catalog.bronze_layer.objects_description_journal")